# RAG Evaluation — GOOD (Improved) VERSION

## Improvements over the bad version:
| Improvement | Impact |
|---|---|
| All **10 topic documents** indexed (covering every test question) | Retriever always finds relevant chunks |
| `similarity_top_k=5` | Rich context for the LLM |
| Small chunks (256 tokens) + overlap (50) | Focused, precise, continuous chunks |
| **Sentence Window Index** (window_size=3) | Retrieves surrounding sentences for richer context |
| `MetadataReplacementPostProcessor` | Expands retrieved sentence to full window |

**Expected scores:** Context Relevance > 0.6, Groundedness > 0.6, Answer Relevance > 0.7

In [1]:
import nest_asyncio
nest_asyncio.apply()

import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
if not GOOGLE_API_KEY or GOOGLE_API_KEY == "your_google_api_key_here":
    raise ValueError("Set GOOGLE_API_KEY in your .env file!")
print("API Key loaded.")

API Key loaded.


## GOOD CHOICE 1: All 10 topic documents — full coverage

In [2]:
from llama_index.core import SimpleDirectoryReader

# GOOD: Load all 10 topic documents from the local data directory
good_documents = SimpleDirectoryReader(input_dir='data').load_data()

total_chars = sum(len(d.text) for d in good_documents)
print(f"GOOD: {len(good_documents)} documents loaded from 'data/' folder, {total_chars:,} total chars — covers all 10 test questions!")


Loaded: Effects of climate change on oceans (26,751 chars)
Loaded: Marine heat wave (10,155 chars)
Loaded: Ocean heat content (16,999 chars)
Loaded: Climate change and ecosystems (19,717 chars)
Loaded: Overfishing (33,216 chars)
Loaded: Ocean acidification (51,548 chars)
Loaded: Marine biodiversity (93,241 chars)
Loaded: Coastal management (34,484 chars)
Loaded: Marine protected area (39,757 chars)
Loaded: Paleoclimatology (26,918 chars)

GOOD: 10 documents, 352,786 total chars — covers all 10 test questions!


##  GOOD CHOICE 2: Small chunks + overlap

In [3]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings

llm = GoogleGenAI(model="gemini-2.5-flash", api_key=GOOGLE_API_KEY, temperature=0.1)
embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001", api_key=GOOGLE_API_KEY)

Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 256   # GOOD: small, focused chunks
Settings.chunk_overlap = 50 # GOOD: overlap keeps context continuous

print("GOOD settings: chunk_size=256, chunk_overlap=50")

/Users/ratneshsingh/Developer/trulens/trulens-eval-demo/venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


GOOD settings: chunk_size=256, chunk_overlap=50


##  GOOD CHOICE 3: Sentence Window Index

Indexes each sentence individually (for precise retrieval) but **expands** the retrieved node to include surrounding sentences (for rich context).

In [4]:
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceWindowNodeParser

node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,                              # GOOD: 3 sentences before + after match
    window_metadata_key="window",
    original_text_metadata_key="original-text"
)

original_parser = Settings.node_parser
Settings.node_parser = node_parser
try:
    good_index = VectorStoreIndex.from_documents(good_documents, show_progress=True)
finally:
    Settings.node_parser = original_parser

print("GOOD: Sentence Window Index built over all 10 topic documents.")

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/515 [00:00<?, ?it/s]

GOOD: Sentence Window Index built over all 10 topic documents.


## GOOD CHOICE 4: top_k=5 + MetadataReplacementPostProcessor

In [5]:
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

good_engine = good_index.as_query_engine(
    similarity_top_k=5,     # GOOD: retrieve 5 chunks (vs 1 in bad)
    node_postprocessors=[
        # GOOD: replaces each matched sentence with its full window context
        MetadataReplacementPostProcessor(target_metadata_key="window")
    ]
)

print("GOOD pipeline: chunk_size=256, overlap=50, Sentence Window, top_k=5")

GOOD pipeline: chunk_size=256, overlap=50, Sentence Window, top_k=5


## Configure TruLens

In [6]:
from trulens.core import TruSession, Metric
from trulens.apps.llamaindex import TruLlama
from trulens.providers.litellm import LiteLLM
from trulens.feedback.groundtruth import GroundTruthAgreement

session = TruSession()
session.reset_database()

os.environ["GEMINI_API_KEY"] = GOOGLE_API_KEY
gemini_provider = LiteLLM(model_engine="gemini/gemini-2.5-flash")

qa_df = pd.read_csv("ipcc_test_questions.csv")
qa_set = [{"query": item["Question"], "response": item["Answer"]} for _, item in qa_df.iterrows()]

m_answer_relevance = Metric(implementation=gemini_provider.relevance_with_cot_reasons, name="Answer Relevance").on_input_output()
m_context_relevance = Metric(implementation=gemini_provider.context_relevance_with_cot_reasons, name="Context Relevance").on_prompt().on_context(collect_list=True)
m_groundedness = Metric(implementation=gemini_provider.groundedness_measure_with_cot_reasons, name="Groundedness").on_context(collect_list=True).on_response()
m_groundtruth = Metric(implementation=GroundTruthAgreement(qa_set, provider=gemini_provider).agreement_measure, name="Answer Correctness").on_input_output()

metrics = [m_answer_relevance, m_context_relevance, m_groundedness, m_groundtruth]

good_recorder = TruLlama(good_engine, app_name="GOOD RAG", feedbacks=metrics)
print(f"TruLens ready. Running {len(qa_set)} queries...")

Package jsonschema not present in requirements.


instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index.core.base.embeddings.base.BaseEmbedding'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index.core.schema.TransformComponent'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index.core.schema.BaseComponent'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'pydantic.main.BaseModel'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index_instrumentation.DispatcherSpanMixin'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class '

/Users/ratneshsingh/Developer/trulens/trulens-eval-demo/venv/lib/python3.14/site-packages/trulens/feedback/llm_provider.py:2857: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  warnings.warn(
WARNI [trulens.feedback.computer] feedback_name=Answer Correctness, record=7832d13b-edd2-431c-a5a6-c0f4751f58d6, span_group=None had an error during computation:
'expected_response'
/Users/ratneshsingh/Developer/trulens/trulens-eval-demo/venv/lib/python3.14/site-packages/trulens/feedback/llm_provider.py:2857: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  warnings.warn(
WARNI [trulens.feedback.computer] feedback_name=Answer Correctness, record=11562c4e-b158-47fe-a123-aeeafbd5afd1, span_group=None had an error during computation:
'expected_response'
/Users/ratneshsingh/Developer/trulens/trulens-eval-demo/venv/lib/python3.14/site-packages/trulens/feedback/llm_provider.py:2857: UserWarning: Failed to proc

## Run Evaluation

In [7]:
print(f"Running {len(qa_set)} queries through GOOD RAG...")
with good_recorder as recording:
    for i, q in enumerate(qa_set):
        print(f"  [{i+1}/{len(qa_set)}] {q['query'][:65]}")
        good_engine.query(q['query'])
print("\nGOOD RAG evaluation complete.")

Running 10 queries through GOOD RAG...
  [1/10] What are the primary impacts of climate change on ocean and coast
  [2/10] How do marine heatwaves affect ocean ecosystems?
  [3/10] What role does the ocean play in global climate regulation?
  [4/10] How is climate change impacting marine biodiversity?
  [5/10] What are the major non-climate drivers affecting ocean and coasta
  [6/10] How does climate change influence the distribution of marine spec
  [7/10] What are the effects of ocean acidification on marine life?
  [8/10] How does climate change affect coastal communities and industries
  [9/10] What are some adaptation strategies for managing climate change i
  [10/10] What is the significance of the paleorecord in understanding clim

GOOD RAG evaluation complete.


In [8]:
records, feedback = session.get_records_and_feedback(app_ids=[])
print(f"Total records: {len(records)}\n")

metric_cols = [c for c in records.columns if c in ["Answer Relevance", "Context Relevance", "Groundedness", "Answer Correctness"]]
print("=== GOOD RAG Scores ===")
for col in metric_cols:
    score = records[col].mean()
    status = "🛑 Poor" if score < 0.4 else "🟡 Mediocre" if score < 0.6 else "🟢 Good" if score < 0.8 else "🌟 Excellent"
    print(f"  {col:25s}: {score:.3f}  {status}")

records[metric_cols + ['app_name']].head(10)

Total records: 9

=== GOOD RAG Scores ===
  Answer Relevance         : 1.000  🌟 Excellent
  Context Relevance        : 1.000  🌟 Excellent
  Groundedness             : 1.000  🌟 Excellent
  Answer Correctness       : nan  🌟 Excellent


,Answer Relevance,Context Relevance,Groundedness,Answer Correctness,app_name
0,1.0,1.0,1.0,NaN,GOOD RAG
1,NaN,NaN,NaN,NaN,GOOD RAG
2,NaN,NaN,NaN,NaN,GOOD RAG
3,1.0,1.0,NaN,NaN,GOOD RAG
4,NaN,NaN,NaN,NaN,GOOD RAG
5,NaN,NaN,NaN,NaN,GOOD RAG
6,NaN,NaN,NaN,NaN,GOOD RAG
7,NaN,NaN,NaN,NaN,GOOD RAG
8,NaN,NaN,NaN,NaN,GOOD RAG


In [9]:
# Dashboard at http://localhost:8501 — scores should be significantly higher!
session.run_dashboard()

Starting dashboard ...


/var/folders/kr/732g325n4j9czjh939zyww3h0000gn/T/ipykernel_97532/295487288.py:2: DeprecationWarning: Method `run_dashboard` has been renamed or moved to `trulens.dashboard.run.run_dashboard`.

  session.run_dashboard()


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://localhost:63367 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>